In [3]:
import cupy as xp
words = open('names.txt').read().splitlines()   # from github.com/karpathy/makemore
assert not any(' ' in w for w in words)

chars = sorted(set(''.join(words)))
stoi = {c: i+1 for i, c in enumerate(chars)}
stoi[' '] = 0
itos = {i: c for c, i in stoi.items()}
vocab_size = len(stoi)                          # 27

context_len = 3
X_list, y_list = [], []
for w in words:
    context = [0] * context_len
    for ch in w + ' ':
        X_list.append(context.copy())
        y_list.append(stoi[ch])
        context = context[1:] + [stoi[ch]]
X = xp.asarray(X_list) / 26.0
X_oh = xp.eye(vocab_size)[xp.asarray(X_list)].reshape(len(X_list), -1)   # (228146, 81) on GPU
Y = xp.eye(vocab_size)[xp.asarray(y_list)]

In [4]:
class Affine:
    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def forward(self, x):
        return x @ self.weight.T + self.bias



In [5]:
class Sigmoid:
    def __init__(self):
        pass

    def forward(self, x):
        return 1 / (xp.exp(-x) + 1)


In [6]:
class Softmax:
  def __init__(self):
    pass

  def forward(self, x):
    e = xp.exp(x - xp.max(x, axis=1, keepdims=True))
    return e / xp.sum(e, axis=1, keepdims=True)



In [16]:
class MLP_name:
    def __init__(self, arc, out):
        self.g = [None] * len(arc)
        for i in range(len(arc)):
            if(i != len(arc) - 1):
                self.g[i] = [xp.random.randn(arc[i + 1], arc[i]) * xp.sqrt(1/arc[i]), xp.zeros((1, arc[i + 1]))]
            else:
                self.g[i] = [xp.random.randn(out, arc[i]) * xp.sqrt(1/arc[i]), xp.zeros((1, out))]
        self.update()
    def forward(self, x):
        for layer in self.layers:
            for module in layer:
                x =  module.forward(x)
        return x.astype(float)
    def update(self):
        self.n = len(self.g)
        self.affine = [None] * self.n
        self.activation = [None] * self.n
        self.layer = [None] * self.n
        self.layers = [None] * self.n

        for i in range(self.n):
            self.affine[i] = Affine(self.g[i][0], self.g[i][1])
            if(i != self.n - 1):
                self.activation[i] = Sigmoid()
            else:
                self.activation[i] = Softmax()
            self.layer[i] = [self.affine[i], self.activation[i]]
            self.layers[i] = (self.layer[i])



    def gd(self, X_train, y_train, lr):
        self.dgd(X_train, y_train, lr)


    def train(self, X_train, y_train, epochs, lr):
        for i in range(epochs):
            self.gd(X_train, y_train, lr)


    def test(self, X_test, y_test):
        preds = xp.argmax(self.forward(X_test), axis=1)
        return float((preds == y_test).mean())



    def dgd(self, x, real, lr):
        xc = x
        n = len(x)
        ins = []
        outs = [0]
        for layer in self.layers:
            ins.append(x)
            for module in layer:
                x =  module.forward(x)
            outs.append(x)
        diff = x - real

        rolling = xp.sqrt(diff)
        for i in range(self.n - 1, -1, -1):
            self.g[i][1] -= lr / n * rolling.sum(axis=0)
            ww = self.g[i][0].copy()
            self.g[i][0] -= lr / n * (rolling.T @  ins[i])
            rolling = rolling @ ww
            if i > 0:
                rolling *= (outs[i]) * (1 - outs[i])
        return None



    def __call__(self, x):
        return self.forward(x)

In [17]:
def top_k(q, k=27):
    idx = xp.argsort(q)[-k:]
    p = xp.zeros_like(q)
    p[idx] = q[idx]
    return p / p.sum()

In [18]:
model2 = MLP_name([81, 128, 128], 27)

model2.train(X_oh, Y, epochs=1000, lr=0.1)





In [19]:
def one_hot(idx, size=27):
    v = xp.zeros(size)
    v[idx] = 1.0
    return v

def un_hot(v):
    return int(xp.argmax(v))

In [21]:
for i in range(100):
    st = ''
    s = [0, 0, 0]
    while True:
        x = model2.forward(xp.eye(27)[xp.asarray(s)].reshape(1, -1))
        #print(x)
        q = xp.asnumpy(x)[0]
        q = q / q.sum()

        l = int(xp.random.choice(27, size=1, p=top_k(q))[0])
        if l == 0:
            break
        s = s[1:] + [l]
        st += itos[l]
    if(len(st) >= 3):
        print(st)


ValueError: probabilities are not non-negative

In [ ]:
model = MLP_name([3, 128, 128], 27)

model.train(X, Y, epochs=1000, lr=0.1)





In [ ]:
st = ''
s = [0, 0, 0]
for i in range(100):
    st = ''
    s = [0, 0, 0]
    while True:
        x = model.forward(xp.asarray(s))
        #print(x)
        q = xp.asnumpy(x)[0]
        q = q / q.sum()

        l = int(xp.random.choice(27, size=1, p=top_k(q))[0])
        if l == 0:
            break
        s = s[1:] + [l]
        st += itos[l]
    if(len(st) >= 3):
        print(st)


In [ ]:
y = xp.asarray(y_list)
print("old:", model.test(X, y))          # /26 model
print("new:", model2.test(X_oh, y))      # one-hot model